# LAB 04

In [1]:
import pygame
import math
import numpy as np

LARGURA, ALTURA = 900, 650
FPS = 60

class Robo:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)
        self.theta = 0.0

        self.angulos = np.radians([-60, -30, 0, 30, 60])
        self.alcance = 150.0
        self.leituras = [self.alcance] * 5

        self.base = 70.0
        self.ganho = 65.0
        self.distancia_critica = 38.0
        self.distancia_rodas = 34.0

    def medir(self, obstaculos):
        leituras = []

        for beta in self.angulos:
            angulo = self.theta + beta
            distancia = self.alcance

            for passo in range(2, int(self.alcance) + 1, 2):
                rx = self.x + passo * math.cos(angulo)
                ry = self.y + passo * math.sin(angulo)

                if rx <= 5 or rx >= LARGURA - 5 or ry <= 5 or ry >= ALTURA - 5:
                    distancia = float(passo)
                    break

                if any(obstaculo.collidepoint(rx, ry) for obstaculo in obstaculos):
                    distancia = float(passo)
                    break

            leituras.append(distancia)

        self.leituras = leituras

    def controlar(self):
        proximidade = [
            1 - min(distancia, self.alcance) / self.alcance
            for distancia in self.leituras
        ]

        esquerda = 0.7 * proximidade[0] + proximidade[1]
        direita = proximidade[3] + 0.7 * proximidade[4]

        vl = self.base + self.ganho * direita
        vr = self.base + self.ganho * esquerda

        if self.leituras[2] < self.distancia_critica:
            soma_esquerda = self.leituras[0] + self.leituras[1]
            soma_direita = self.leituras[3] + self.leituras[4]

            if soma_esquerda < soma_direita:
                vl, vr = 75, -75
            else:
                vl, vr = -75, 75

        return vl, vr

    def atualizar(self, dt, vl, vr):
        v = (vl + vr) / 2
        omega = (vr - vl) / self.distancia_rodas

        self.theta += omega * dt

        self.x += v * math.cos(self.theta) * dt
        self.y += v * math.sin(self.theta) * dt

        self.x = float(np.clip(self.x, 15, LARGURA - 15))
        self.y = float(np.clip(self.y, 15, ALTURA - 15))

    def desenhar(self, tela):
        pygame.draw.circle(tela, (0, 200, 255), (int(self.x), int(self.y)), 14)

        for beta, distancia in zip(self.angulos, self.leituras):
            angulo = self.theta + beta

            ex = self.x + distancia * math.cos(angulo)
            ey = self.y + distancia * math.sin(angulo)

            pygame.draw.line(tela, (255, 210, 0), (self.x, self.y), (ex, ey), 1)

def main():
    pygame.init()

    tela = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB 04 - Braitenberg")
    relogio = pygame.time.Clock()
    fonte = pygame.font.SysFont("monospace", 14)

    robo = Robo(90, 500)

    obstaculos = [
        pygame.Rect(180, 120, 100, 260),
        pygame.Rect(390, 320, 150, 100),
        pygame.Rect(600, 100, 100, 280),
        pygame.Rect(680, 470, 130, 80)
    ]

    rodando = True

    while rodando:
        dt = relogio.tick(FPS) / 1000

        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False

        robo.medir(obstaculos)

        vl, vr = robo.controlar()

        robo.atualizar(dt, vl, vr)

        tela.fill((20, 24, 30))

        for obstaculo in obstaculos:
            pygame.draw.rect(tela, (180, 50, 50), obstaculo)

        robo.desenhar(tela)

        texto = fonte.render(
            f"vL={vl:.1f}  vR={vr:.1f}  frente={robo.leituras[2]:.1f}px",
            True,
            (240, 240, 240)
        )

        tela.blit(texto, (20, 20))

        pygame.display.flip()

    pygame.quit()

main()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
